In [ ]:
## Import module dependencies
import pandas as pd #for manipulating data tables and csv files
import numpy as np #for mathematical functions
import matplotlib.pyplot as plt #for graphical plotting 
from matplotlib import colormaps

## Importing various items from scikitlearn - a more intuitive, introductory toolbox for Machine Learning in Python

## For acquiring and preprocessing data used in this tutorial
from sklearn.datasets import fetch_openml, load_iris
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split

## Importing ML models used for doing clustering
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.ensemble import RandomForestClassifier

# Before we get started let's define some functions to make life a little easier

## First, let is define a function for plotting the dataset:

In [ ]:
def plot_dataset(dataset, feature_names, colorsort, nsorted, sort_labels, 
                 sort_title="categories", cluster_centres=None):

    fig, axs = plt.subplots(
        len(feature_names), len(feature_names), figsize=(100,40))

    axs = axs.T
    for x in range(len(feature_names)):
        x_label = feature_names[x]
        axs[x,-1].set_xlabel(x_label, size=40)
        for y in range(len(feature_names)):
            if x<y:
                y_label = feature_names[y]
                colors = ["C"+str(c) for c in dataset[colorsort]]
                color_centres = ["C"+str(c) for c in range(0, nsorted)]
                axs[x,y].scatter(dataset[x_label][:], dataset[y_label][:], c=colors, alpha=0.5, s=100)
                if x_label in dataset and y_label in dataset and cluster_centres is not None:
                    axs[x,y].scatter(cluster_centres[x_label], cluster_centres[y_label], marker="^", c=color_centres, s=300)
                axs[0,y].set_ylabel(y_label, size=40)
            elif x==y:
                plotData = [dataset.loc[(dataset[colorsort]==c)][x_label] for c in range(0, nsorted)]
                colors = ["C"+str(c) for c in range(0, nsorted)]
                axs[x,y].hist(plotData, stacked=True, label=sort_labels, color=colors)
                axs[x,y].set_ylabel(sort_title, size=40)
                axs[x,y].legend(fontsize=30)
            else:
                axs[x,y].set_visible(False)
    plt.show()

## Now let's define some functions for assessing the clustering, both graphically and analytically

In [ ]:
def find_cluster_centres(dataset, kmeans_model):
    cluster_centres = {}
    for a,b in zip(list(dataset), kmeans_model.cluster_centers_.T):
        cluster_centres[a] = b
    newdataset = dataset.copy()
    newdataset["cluster"] = kmeans_model.predict(dataset)
    
    return newdataset, cluster_centres

In [ ]:
def assess_clustering(dataset, clustercol, targetcol, nclusters, ntargets, target_names, targetAxTitle):
    fig, ax = plt.subplots(2,1, figsize=(5,10))

    hist, xbins, ybins, im = ax[0].hist2d(dataset[clustercol], dataset[targetcol], bins=(ntargets,nclusters))
    ax[0].set_yticks(np.arange((ntargets-1)/(2*ntargets), ntargets-1, (ntargets-1)/ntargets), labels=target_names)
    ax[0].set_xticks(np.arange((nclusters-1)/(2*nclusters), nclusters-1, (nclusters-1)/nclusters), labels=np.arange(0, nclusters, 1))

    ax[0].set_xlabel("cluster")
    ax[0].set_ylabel(targetAxTitle)

    fig.colorbar(im)
    gini_impurity = np.zeros(nclusters)
    for i in range(len(xbins)-1):
        n_in_clust = sum(hist[i])
        gini = 1.001
        for j in range(len(ybins)-1):
            gini -= (hist[i,j]/n_in_clust)**2
            ax[0].text(xbins[j]+0.5,ybins[i]+0.5, "%0.f" %hist.T[i,j], 
                color="w", ha="center", va="center", fontweight="bold")
        gini_impurity[i] = gini
        

    ax[1].bar(np.arange(0, nclusters, 1), gini_impurity, width=0.8, align="center", color=gini_impurity)
    ax[1].set_xlabel("cluster")
    ax[1].set_xticks(np.arange(0, nclusters, 1), labels=np.arange(0, nclusters, 1))
    ax[1].set_ylabel("gini impurity")
    plt.show()

# In this notebook, we shall use the iris dataset to investigate the use of clustering machine learning to solve problems.

## The data is made up of two parts

- Input *features* (e.g. sepal width, petal width)
- Output *target*, in this case a three-way classification between three different iris species (virginica, versicolor, and setosa)

## The aim of a clustering problem is one which can be formulated thusly:
### Given data with some given *features*, can we assign these data into groups or *clusters* such that we learn something about patterns or structure in the data.

In our case, this means we want to define a sorting which puts an iris into a cluster based on its features. The question we want to investigate is therefore

### How well does an unsupervised clustering or sorting compare to the defined species assigned to the irises.

# Some other things to consider:
## How do we ensure our models and their predictions are robust?
## What are some things we can do to boost our confidence in our machine learning models?

# Let's get started by loading the iris dataset.

- 'sepal length (cm)',
- 'sepal width (cm)',
- 'petal length (cm)',
- 'petal width (cm)'

In [ ]:
iris_dataset = load_iris(as_frame=True)

iris_data = iris_dataset["data"]
class_names = iris_dataset["target_names"]
feature_names = iris_dataset["feature_names"]

In [ ]:
#1) Define the model
kmeans = KMeans(n_clusters=3, random_state=0, n_init="auto", algorithm="elkan")

#2)Train the model
kmeans.fit(iris_data)

In [ ]:
#3) Populate the data with target (species) and assigned clustering from the model
sorted_data, cluster_centres = find_cluster_centres(iris_data, kmeans)
sorted_data["target"] = iris_dataset["target"]

# In the above, we have loaded the iris dataset and trained a K-Means clustering algorithm on its features

## K-Means clustering

Given some data in a feature space (e.g. in 2 dimensions, points on a plot of `(x1, x2)`), k-means clustering partitions the data into *k* non-overlapping sub-groups by optimising **cluster centroids**. These can be thought of as the centre of the clusters. These are defined and optimised in training by minimising the distance between data points and the centroids.
An example is given below for *k=3* clusters in a 2D feature space:

![](kmeans.png)
An example of *k=3* k-means clustering in a feature space of 2-dimensions. Image taken from bookdown.org k-means-clustering.

## Other clustering algorithms

### Not all algorithms are created equal!
Depending on how one may expect data to be distributed within some feature space and depending on the kind of question one wants answering, different clustering algorithms may be more appropriate than others. An illustration of this is shown below where the number of clusters is minimised to satisfy the optimisation condition of the algorithm.

![](clustering_algorithms.png)
A comparison of various clustering algorithms and how they perform on various data distributions within a feature space. The times shown indicate the duration of execution. Image taken from https://scikit-learn.org. 

\
Although such intuitions may not apply to very high-dimensional data, they may be a good starting point when considering the best algorithm(s) for a given problem. Different use cases of clustering may be
1. Identifying trend groups in data, perhaps for subsequent granular analysis,
2. Describing "ordinary" clusters of data so that outliers or anomalies can later be detected,
3. Simplifying higher-dimension data to aid in subsequent tasks such as classification.

These use cases may look quite different for different real-world data examples:
- Customer, client, or patient segmentation in marketing or healthcare.
- Image segmentation to identify objects of interest in medicine or biology.
- Document segmentation to identify topics of interest.
- Analysis of social networks.
- Anomaly detection in web-security or banking.

## Have a play!

In the steps above, feel free to investigate what happens when you change the `algorithm` or other parameters in the `KMeans` initialisation call. It may be worth reading up on documentation available [here](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) to see what options are available.

You may also wish to investigate using `AgglomerativeClustering` instead. This is imported in this workbook and can be thought of as an "inverse" or bottom-up approach compared to KMeans. This is because this model works by first defining **every** data point as its own cluster then iteratively merging the closest pairs using a linkage distance until all data points belong to one of the large clusters. A diagramatic for this is shown below:

![](agglomerative_clustering_compact_.webp)
An illustrative example of agglomerative clustering where a range of animals (each their own cluster initially) are iteratively merged into larger clusters.

# Visualising the clustering

The previously defined `plot_dataset` function can now be used to examine our data. By calling it, the Iris dataset can be viewed across all its available input features, both in 1D histograms and 2D scatter plots for each possible pair of features. 

## First, let's look at the data separated by the *known* species categorisation:

In [ ]:
plot_dataset(sorted_data, feature_names, "target", 3, class_names, sort_title="categories")

Looking across the available plot visualisations above, three points should be clear:
1. Some features provide better separation between pairs of categories than others
2. *Setosa* Irises are generally better separated than the other two species, which show a higher degree of overlap in the data
3. Features show varying degrees of positive and negative correlation

In general, such "messiness" in data is guaranteed. Depending on the scenario, this may be very large or it may be very small. However, it is important to recognise this to understand the limitations of statistics and machine learning.

## Then, let's look at the data separated by the *unsupervised clustering* trained previously:

In [ ]:
plot_dataset(sorted_data, feature_names, "cluster", 3, (0,1,2) , sort_title="clustering", cluster_centres=cluster_centres)

# Interpreting the results

While it is not always possible to compare the results of a clustering to some known categorisation, the Iris dataset is an example where this *can* be done - hopefully to show how the results of clustering may be interpreted in broader examples.

Examining the above by eye, it may appear that the clustering algorithm has some kind of correspondance to the known species categorisation. This is reassuring in the sense that the clustering can provide something relatively intuitive for further interpretation. Moreover, in *this* examples, it suggests that the clustering *may* be a useful tool for, say, determining a method to categorise irises into their species based on their features. 

## Let's make some plots to let us get into more detail

### Confusion matrix

It can also be described as an overlap matrix. On one axis is the assigned cluster while on the other is the known species. In the case of wanting clustering which can help us to perform categorisation of the data, the most desirable case is for every datapoint in any row (column) to sit in only one cell. This corresponds to a perfect separation and would mean that every cluster has a perfect one-to-one correspondence with one of the species. 
Unfortunately, no real, finite dataset would produce this kind of outcome. However, it may be possible to get very close.

#### Depending on the particular scenario and the objective of a cluster-based analysis
Some overlaps may or may not be more acceptable than others. In this case, the below plot shows that cluster *0* contains both *virginica* and *versicolor* irises. If the goal was to use the clustering for categorising specifically, say, *setosa* irises from all others, this may be fine. Alternatively, if the aim was to identify close to *100\%* of *virginica* irises, this *may* be a problem.

### Gini impurity
This metric describes the "impurity" of some given set of data. It is usually interpreted as the probability that some random datapoint within a set is incorrectly identified as belonging to an assigned category. It is commonly used in classification and regression but it is also useful here for describing the "usefulness" of clusters assuming they may be used as classes. 

In [ ]:
assess_clustering(sorted_data, "cluster", "target", 3, 3, class_names, "species")

These metrics support the earlier thinking that the clusters may have some mapping against the known species. However, they contextualise this with an assessment of the clusters' purity to allow us to understand the limitations of such usage. 

## Have a play!

Before moving on, consider what happens if you
1) Change the hyperparameters of the clustering model
2) Switch between various models
3) Change the number of clusters - what about 2 clusters? Or 5? Why might we not want to have *too many* clusters optimised in the model?

# Using the clustering for classification

Now that we have done **unsupervised** training on the data to define clusters, we can now consider what happens if we use the results of this as input in **supervised** classification.
While this is shown only briefly using Random Forest classifier, you may wish to look at a tutorial on classification linked below for further discussion.



### Classification Tutorial: [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/cly20/ml_classification_tutorial/HEAD?urlpath=%2Fdoc%2Ftree%2FTitanicClassification.ipynb)

In [ ]:
#0) Prepare the dataset (with clustering info) and split between training and testing samples
ForClassify = sorted_data.drop("target", axis=1)
X_train, X_test, Y_train, Y_test = train_test_split(ForClassify, iris_dataset["target"])

#1) Define and train model
RF_classifier = RandomForestClassifier()
RF_classifier.fit(X_train, Y_train)
print("Training score is ", RF_classifier.score(X_train, Y_train))

#2) Test model
print("Testing score is ", RF_classifier.score(X_test, Y_test))

Already this shows that clustering can provide a very powerful handle for being able to do classification. However, to do this robustly and confidently, examination of metrics such as ROC curves, confusion matrices would be needed. Stronger modelling may also be possible by looking into the choice of model(s) and their hyperparameters.

# Over to you!!

## Best of luck!